# Artefato — coleta de metadados no OpenAlex

Recuperado do commit `3ca0d72883c8154f4b95c3d94d9686b06f2d6674`, arquivo `01_limpeza/Enriquecimento/OpenAlex.ipynb`.

Este é o coletor original preservado no Git. A reconstrução do resumo pelo índice invertido, a extração de autores/data/título, as tentativas e os status foram mantidos. Foram adaptados os caminhos para a organização atual, removido o contato pessoal e adicionada uma opção de execução desativada por padrão.

Não integra os seis notebooks do pipeline principal. A coleta não foi executada nesta revisão; apenas o modo desativado foi validado. O fluxo principal apenas lê o checkpoint já fornecido. Executar todas as células com `EXECUTAR_CONSULTAS=False` não faz requisições nem grava dados. Os parâmetros da coleta representam a implementação original.

[Documentação atual do OpenAlex](https://help.openalex.org/api/) · [Campos bibliográficos](https://help.openalex.org/data/works/attributes/)


# Consulta OpenAlex API via DOI

Este notebook consulta a API [OpenAlex](https://api.openalex.org) para enriquecer registros incompletos do dataset CORD-19 usando seus DOIs.

A OpenAlex é uma API gratuita e aberta que fornece metadados de publicações acadêmicas, incluindo título, abstract, autores e data de publicação.

**Fluxo:**
- Carregar as instâncias incompletas (`metadata_incompletas.csv`)
- Consultar a OpenAlex API para cada DOI
- Preencher os campos faltantes com os dados obtidos
- Salvar o resultado enriquecido (`metadata_enriquecidas.csv`)

In [ ]:
import os
import time
import json
import requests
import pandas as pd
from pathlib import Path
from IPython.display import display, clear_output

EXECUTAR_CONSULTAS = False
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/config.py").is_file())
PASTA_DADOS = RAIZ / "data/staged"
CAMINHO_INCOMPLETAS = PASTA_DADOS / "01_incompletas.parquet"
if EXECUTAR_CONSULTAS:
    df_incompletas = pd.read_parquet(CAMINHO_INCOMPLETAS)
    display(df_incompletas.isna().sum())


## Carregando instâncias incompletas

In [ ]:
if EXECUTAR_CONSULTAS:
    CAMINHO_INCOMPLETAS = PASTA_DADOS / "metadata_incompletas.csv"
    df_incompletas = pd.read_csv(CAMINHO_INCOMPLETAS, dtype=str)
    print(f"Instâncias incompletas carregadas: {len(df_incompletas):,}")
    print(f"\nNulos por coluna:")
    print(df_incompletas.isna().sum())
    display(df_incompletas.head())

## Função de consulta à API OpenAlex

- **Endpoint:** `https://api.openalex.org/works/doi:{doi}`
- **Campos retornados:** título, abstract, autores, data de publicação

In [ ]:
if EXECUTAR_CONSULTAS:
    # Consulta a API OpenAlex para obter metadados de um artigo pelo DOI.
    def consultar_openalex(doi, timeout=10):
        url = f"https://api.openalex.org/works/doi:{doi}"
        headers = {"User-Agent": os.getenv("OPENALEX_USER_AGENT", "TCC-CORD19")}
    
        try:
            resp = requests.get(url, headers=headers, timeout=timeout)
            if resp.status_code == 200:
                data = resp.json()
                resultado = {}
            
                # Título
                titulo = data.get("title")
                if titulo:
                    resultado["title"] = titulo
            
                # Abstract (OpenAlex retorna abstract como inverted index)
                abstract_inv = data.get("abstract_inverted_index")
                if abstract_inv:
                    # Reconstruir abstract a partir do inverted index
                    palavras = {}
                    for palavra, posicoes in abstract_inv.items():
                        for pos in posicoes:
                            palavras[pos] = palavra
                    abstract_texto = " ".join(palavras[k] for k in sorted(palavras.keys()))
                    resultado["abstract"] = abstract_texto
            
                # Autores
                autores_raw = data.get("authorships", [])
                if autores_raw:
                    nomes = [a.get("author", {}).get("display_name", "") for a in autores_raw]
                    nomes = [n for n in nomes if n]
                    if nomes:
                        resultado["authors"] = "; ".join(nomes)
            
                # Data de publicação
                pub_date = data.get("publication_date")
                if pub_date:
                    resultado["publish_time"] = pub_date
            
                return resultado
            elif resp.status_code == 404:
                return {}  # DOI não encontrado
            else:
                return None  # Erro, tentar novamente
        except requests.exceptions.RequestException:
            return None  # Erro de rede

    # Teste com um DOI conhecido
    teste = consultar_openalex("10.1186/1471-2334-1-6")
    print("Teste de consulta:")
    for k, v in teste.items():
        print(f"  {k}: {str(v)[:100]}..." if len(str(v)) > 100 else f"  {k}: {v}")

## Enriquecimento em lote

O processamento é feito em lote com:
- **Checkpoints** salvos periodicamente para poder retomar de onde parou
- **Rate limiting** para respeitar os limites da API
- **Retries** em caso de erros de rede

In [ ]:
if EXECUTAR_CONSULTAS:
    # Configurações do processamento em lote
    TAMANHO_LOTE = 100  # Salvar checkpoint a cada N consultas
    INTERVALO_ENTRE_REQUESTS = 0.0125  # Segundos entre requisições (ser educado com a API)
    MAX_TENTATIVAS = 3  # Tentativas máximas por DOI em caso de erro

    # Arquivo de checkpoint para retomar de onde parou
    PASTA_ENRIQUECIMENTO = RAIZ / "data/checkpoints/recuperacao_dados"
    CAMINHO_CHECKPOINT = PASTA_ENRIQUECIMENTO / "checkpoint_enriquecimento.json"

    # Carregar checkpoint se existir
    resultados = {}
    if CAMINHO_CHECKPOINT.exists():
        with open(CAMINHO_CHECKPOINT, "r", encoding="utf-8") as f:
            resultados = json.load(f)
        print(f"Checkpoint carregado: {len(resultados):,} DOIs já consultados")
    else:
        print("Nenhum checkpoint encontrado. Iniciando do zero.")

    # DOIs a consultar (apenas os que ainda não foram consultados)
    dois_para_consultar = [
        doi for doi in df_incompletas["doi"].dropna().unique()
        if doi not in resultados
    ]
    print(f"DOIs a consultar: {len(dois_para_consultar):,}")
    print(f"DOIs já consultados: {len(resultados):,}")

In [ ]:
if EXECUTAR_CONSULTAS:
    total = len(dois_para_consultar)
    sucesso = 0
    falhas = 0
    nao_encontrados = 0

    for i, doi in enumerate(dois_para_consultar):
        # Tentar consultar com retries
        resultado = None
        for tentativa in range(MAX_TENTATIVAS):
            resultado = consultar_openalex(doi)
            if resultado is not None:
                break
            time.sleep(1)  # Esperar mais em caso de erro
    
        if resultado is None:
            resultados[doi] = {"_status": "erro"}
            falhas += 1
        elif len(resultado) == 0:
            resultados[doi] = {"_status": "nao_encontrado"}
            nao_encontrados += 1
        else:
            resultado["_status"] = "sucesso"
            resultados[doi] = resultado
            sucesso += 1
    
        # Progresso
        if (i + 1) % 50 == 0 or (i + 1) == total:
            clear_output(wait=True)
            print(f"Progresso: {i+1:,}/{total:,} ({(i+1)/total*100:.1f}%)")
            print(f"  Sucesso: {sucesso:,} | Não encontrado: {nao_encontrados:,} | Erros: {falhas:,}")
    
        # Salvar checkpoint periodicamente
        if (i + 1) % TAMANHO_LOTE == 0:
            with open(CAMINHO_CHECKPOINT, "w", encoding="utf-8") as f:
                json.dump(resultados, f, ensure_ascii=False)
    
        time.sleep(INTERVALO_ENTRE_REQUESTS)

    # Salvar checkpoint final
    with open(CAMINHO_CHECKPOINT, "w", encoding="utf-8") as f:
        json.dump(resultados, f, ensure_ascii=False)

    print(f"\n--- Enriquecimento concluído ---")
    print(f"Total consultado: {total:,}")
    print(f"Sucesso: {sucesso:,}")
    print(f"Não encontrado: {nao_encontrados:,}")
    print(f"Erros: {falhas:,}")

In [ ]:
if EXECUTAR_CONSULTAS:
    resultados

## Aplicando os resultados

In [ ]:
if EXECUTAR_CONSULTAS:
    # Aplicar os resultados ao dataframe de incompletas
    colunas_preencher = ["title", "abstract", "authors", "publish_time"]
    preenchidos = 0

    for idx, row in df_incompletas.iterrows():
        doi = row["doi"]
        if doi in resultados and resultados[doi].get("_status") == "sucesso":
            for col in colunas_preencher:
                if pd.isna(row[col]) and col in resultados[doi]:
                    df_incompletas.at[idx, col] = resultados[doi][col]
                    preenchidos += 1

    print(f"Campos preenchidos: {preenchidos:,}")
    print(f"\nNulos restantes por coluna:")
    print(df_incompletas.isna().sum())

    # Salvar o resultado enriquecido
    CAMINHO_ENRIQUECIDAS = PASTA_DADOS / "metadata_enriquecidas_coleta.csv"
    df_incompletas.to_csv(CAMINHO_ENRIQUECIDAS, index=False, encoding="utf-8")
    print(f"\nDataset enriquecido salvo em: {CAMINHO_ENRIQUECIDAS}")
    print(f"Registros: {len(df_incompletas):,}")